<div style="display:flex; align-items:center; gap:18px; text-align:left">
  <img src="https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/assets/logo_ust.png" width="100">
  <div>
    <p>Diplomado en Inteligencia Artificial para los Negocios</p>
    <p>Facultad de Ingeniería y Negocios</p>
    <p>Módulo 2: Fundamentos de Machine Learning y herramientas Low Code</p>
    <p>Semana 07: Ejercicios de repaso 3 · Clase 7</p>
  </div>
</div>

# Ejercicios de repaso 3 · Clase 7: Introducción a Forecasting

Estos ejercicios repasan de manera práctica la **Clase 7**: cómo trabajar con **series de tiempo**
en formato estándar (`unique_id`, `ds`, `y`), reconocer a ojo la **anatomía** de la serie (tendencia,
estacionalidad y quiebres estructurales), entrenar modelos automáticos con **StatsForecast**, validar
respetando el tiempo (**ventanas móviles**), medir el error con **MAE y MAPE**, comunicar la
**incertidumbre con intervalos** y tomar **decisiones de capacidad ante costos asimétricos**.

Son **actividades formativas**: sirven para practicar, ganar soltura y recibir *feedback*.

**Cómo trabajarlos:**

- Ejecuten primero las celdas de **preparación** y la celda de **datos**.
- En cada ejercicio hay una celda de código con **pistas** como comentarios: casi siempre basta con
  copiar la pista, quitarle el `#` y completarla. El código es corto a propósito (*low-code*).
- Después de cada ejercicio hay una celda **"Su lectura"**: lo central del diplomado no es el código,
  sino **interpretar y decidir**. Respondan ahí con sus palabras.
- Cuando terminen, comparen con el notebook de **solución comentada**: explica por qué se escribió cada
  línea y, en los ejercicios principales, entrega un *prompt* para pedirle a una IA que les ayude a
  corregir su propio código.

## Preparación del entorno

Instalamos `statsforecast` y `utilsforecast` (la salida queda oculta) y dejamos lista la configuración general.

In [ ]:
%%capture
!pip install -q "statsforecast==2.0.3" "utilsforecast==0.2.15" plotly ipywidgets

In [ ]:
%matplotlib inline
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoETS, AutoARIMA
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae, mape

# en Colab los datos se leen del repo público; en local, desde ../datasets/
REPO = 'https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/'
BASE = '../datasets/' if os.path.exists('../datasets') else REPO

## Bloque A · Qué es una serie de tiempo y exploración (la anatomía)

**Caso:** tráfico aéreo de pasajeros en Chile (`trafico_aereo_cl.csv`, datos oficiales de la Junta de
Aeronáutica Civil - JAC). Cada fila registra el número de pasajeros que volaron en un mes determinado.

En una serie de tiempo el **orden de las filas es información**: no se pueden barajar.

In [ ]:
df = pd.read_csv(BASE + 'trafico_aereo_cl.csv', parse_dates=['ds'])
print('Filas y columnas:', df.shape)
df.head()

### Ejercicio A1 · El formato estándar: unique_id, ds, y

Muestren cuántas **series distintas** hay en `unique_id`, cuál es el **rango de fechas**
(fecha mínima y máxima en `ds`) y comprueben si hay valores nulos.
¿Por qué herramientas modernas como StatsForecast exigen este **formato largo** de tres columnas en vez
de una tabla ancha con una columna por serie?

**Criterio de éxito:** identificar las 3 series, el período cubierto (2010 a 2025 mensual) y argumentar la ventaja del formato largo.

In [ ]:
# pista:
# df['unique_id'].value_counts()
# df['ds'].min(), df['ds'].max()
# df.isna().sum()

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio A2 · Leer la anatomía a ojo: tendencia, estacionalidad y quiebres

Dibujen un gráfico de línea interactivo con Plotly de la serie **`total`** a lo largo de todo el tiempo.
Identifiquen a simple vista:
1. La **tendencia** antes del año 2020.
2. La **estacionalidad**: ¿en qué mes del año se produce sistemáticamente el mayor número de pasajeros?
3. El **cambio estructural**: ¿qué ocurrió a partir de marzo de 2020? ¿Puede un modelo de forecasting
    anticipar un evento como ese?

**Criterio de éxito:** gráfico de la serie total e identificación clara de los tres componentes visuales y el límite del modelo.

In [ ]:
# pista:
# serie_total = df[df['unique_id'] == 'total']
# px.line(serie_total, x='ds', y='y', title='Evolución del tráfico aéreo total en Chile')

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

## Bloque B · Baselines y modelos estadísticos automáticos

Un modelo sofisticado solo tiene sentido si **supera a una regla simple**.
En este bloque enfrentamos las líneas base (*baselines*) contra modelos estadísticos automáticos.

### Ejercicio B1 · Las líneas base: Naive vs SeasonalNaive

Filtren la tabla para trabajar con la serie **`nacional`**.
Configuren un objeto `StatsForecast` con frecuencia mensual (`freq='MS'`) que contenga dos *baselines*:
- `Naive()`: predice que el próximo mes será exactamente igual al último mes observado ("mañana igual a hoy").
- `SeasonalNaive(season_length=12)`: predice que el próximo mes será igual al mismo mes del año anterior.
Generen un pronóstico a **12 meses** (`h=12`). ¿Cuál de los dos baselines reproduce las ondas del verano?
¿Por qué la regla del curso dice que cualquier modelo complejo debe ganarle a estas líneas base?

**Criterio de éxito:** ajustar Naive y SeasonalNaive, ver el pronóstico a 12 meses y explicar el rol del baseline.

In [ ]:
# pista:
# df_nac = df[df['unique_id'] == 'nacional']
# sf_base = StatsForecast(models=[Naive(), SeasonalNaive(season_length=12)], freq='MS')
# pron_base = sf_base.forecast(df=df_nac, h=12)
# pron_base.head()

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio B2 · Modelos estadísticos automáticos con StatsForecast

Agreguen al objeto `StatsForecast` los modelos automáticos **`AutoETS(season_length=12)`** (suavizado
exponencial que aprende nivel, tendencia y estacionalidad) y **`AutoARIMA(season_length=12)`** (que analiza
la correlación entre los períodos recientes).
Entrenen los 4 modelos (Naive, SeasonalNaive, AutoETS y AutoARIMA) para la serie nacional en una sola
llamada a `forecast(df=df_nac, h=12)`. Muestren la tabla resultante.
¿Qué significa el prefijo **`Auto`** en estos modelos y por qué es una ventaja para el enfoque *low-code*?

**Criterio de éxito:** generar el pronóstico con los 4 modelos en una sola llamada y explicar el significado de Auto.

In [ ]:
# pista:
# modelos = [Naive(), SeasonalNaive(season_length=12), AutoETS(season_length=12), AutoARIMA(season_length=12)]
# sf = StatsForecast(models=modelos, freq='MS')
# pron_todos = sf.forecast(df=df_nac, h=12)
# pron_todos.head()

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

## Bloque C · Validación temporal y métricas

En Machine Learning tabular barajábamos las filas al azar (k-fold).
En series de tiempo **barajar es trampa**: no podemos usar datos de 2024 para predecir 2023.
En este bloque aprendemos a validar respetando la flecha del tiempo con **ventanas móviles**.

### Ejercicio C1 · Validación con ventanas móviles (prohibido barajar)

Ejecuten la validación cruzada temporal con `sf.cross_validation` sobre la serie nacional para un
horizonte de **12 meses** (`h=12`), usando **2 ventanas** (`n_windows=2`) y un salto de 12 meses
(`step_size=12`).
Muestren las primeras filas del resultado `cv`. Observen la columna `cutoff`: ¿qué representa esa fecha?
¿Por qué este método simula con fidelidad la realidad operativa de un analista?

**Criterio de éxito:** ejecutar cross_validation temporal con 2 ventanas y explicar conceptualmente qué representa el cutoff.

In [ ]:
# pista:
# cv = sf.cross_validation(df=df_nac, h=12, n_windows=2, step_size=12)
# cv.head()

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio C2 · Evaluación del error: MAE y MAPE por modelo

Usando la función `evaluate` de `utilsforecast.evaluation` con las métricas `[mae, mape]`,
calculen el error de cada uno de los 4 modelos en la validación temporal.
Multipliquen el MAPE por 100 para leerlo como porcentaje.
¿Qué modelo obtiene el menor error? ¿Cuánto se equivoca en promedio (en cantidad de pasajeros y en %)?
¿Superaron los modelos automáticos a los baselines?

**Criterio de éxito:** tabla de errores con MAE y MAPE, identificación del ganador e interpretación de los números en lenguaje de negocio.

In [ ]:
# pista:
# nombres_modelos = ['Naive', 'SeasonalNaive', 'AutoETS', 'AutoARIMA']
# tabla_error = evaluate(cv.drop(columns='cutoff'), metrics=[mae, mape], models=nombres_modelos)
# tabla_error.loc[tabla_error['metric'] == 'mape', nombres_modelos] *= 100
# tabla_error

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

## Bloque D · Del pronóstico a la decisión de capacidad e incertidumbre

Un pronóstico puntual casi nunca acierta con exactitud quirúrgica. Por eso, un pronóstico profesional
siempre se comunica con su **rango de incertidumbre** (intervalos de predicción) y se traduce a una
**decisión operativa** considerando las consecuencias de quedarse corto o pasarse de largo.

### Ejercicio D1 · Pronóstico futuro con intervalos de predicción (80% y 95%)

Con el modelo ganador (`AutoETS`), generen el pronóstico a 12 meses hacia el futuro incluyendo
intervalos de predicción al 80% y al 95% (`level=[80, 95]`).
Grafiquen los últimos 3 años de historia junto al pronóstico central y las bandas sombreadas de
incertidumbre.
¿Por qué las bandas se ensanchan a medida que nos alejamos en el tiempo? ¿Por qué esto es un
**intervalo de predicción** y no una promesa matemática infalible?

**Criterio de éxito:** generar el forecast con intervalos level=[80, 95], graficar las bandas y explicar el ensanchamiento.

In [ ]:
# pista:
# sf_ganador = StatsForecast(models=[AutoETS(season_length=12)], freq='MS')
# fc = sf_ganador.forecast(df=df_nac, h=12, level=[80, 95])
# graficar fc['AutoETS'], fc['AutoETS-lo-95'], fc['AutoETS-hi-95']...

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

### Ejercicio D2 · Decisión operativa de capacidad ante costos asimétricos

Imaginen que son los gerentes de operaciones del aeropuerto de Santiago y deben dimensionar los
turnos del personal de control de seguridad (DGAC), counters y mangas de embarque para el primer mes
proyectado.
Miren las cifras de la celda D1: el pronóstico central (~860.000 pasajeros) y el techo del 95% (~900.000 pasajeros).
Analicen:
1. ¿Cuál es el costo de **subestimar la demanda**? (si dimensionan para 860.000 y llegan 900.000).
2. ¿Cuál es el costo de **sobreestimar la demanda**? (si dimensionan para 900.000 y llegan 860.000).
3. Como en un aeropuerto las filas interminables y la pérdida de vuelos provocan un caos reputacional y
   multas graves, ¿con qué cifra de la banda recomendarían planificar la dotación del personal?

**Criterio de éxito:** argumentar una recomendación operativa basada en la asimetría de costos entre quedarse corto y pasarse.

In [ ]:
# pista: no requiere código nuevo; requiere redactar el análisis cuantitativo y cualitativo
# apoyándose en las cifras del primer mes obtenidas en D1.

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

## Bloque E · Límites del forecasting y ética

No todas las series temporales pueden pronosticarse con éxito.
En este bloque cerramos reflexionando sobre la frontera entre la demanda física y los mercados financieros.

### Ejercicio E1 · Por qué pronosticar demanda no es predecir la bolsa

Esta celda compara conceptualmente la serie de pasajeros aéreos con una serie de precios financieros
(como el precio de una acción bursátil o el tipo de cambio).
Ejecútenla y respondan:
1. ¿Por qué los modelos estadísticos funcionan tan bien con la demanda aérea y fallan estrepitosamente al
   intentar predecir el precio futuro de una acción?
2. ¿Por qué las promesas de "algoritmos de IA para hacerse rico en la bolsa" son casi invariablemente un engaño?

**Criterio de éxito:** explicar la diferencia entre procesos estacionales/físicos y paseos aleatorios en mercados eficientes.

In [ ]:
# simulamos una caminata aleatoria típica de una acción bursátil
np.random.seed(42)
pasos = np.random.normal(0, 1, 100)
precio_accion = 100 + np.cumsum(pasos)
fechas_sim = pd.date_range('2024-01-01', periods=100, freq='B')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 1. demanda física: ciclos humanos y estacionalidad repetible
ax1.plot(hist.index[-24:], hist.values[-24:], color='#004F45', lw=2)
ax1.set_title('Demanda física (Pasajeros aéreos)\nPatrón repetitivo según el calendario')
ax1.set_ylabel('Pasajeros'); ax1.tick_params(axis='x', rotation=45)

# 2. precio financiero: paseo aleatorio dominado por noticias nuevas
ax2.plot(fechas_sim, precio_accion, color='#E45756', lw=2)
ax2.set_title('Activo financiero (Acción bursátil)\nPaseo aleatorio: el pasado no predice sorpresas')
ax2.set_ylabel('Precio ($)'); ax2.tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

**Su lectura:** _escriban aquí su interpretación, con sus palabras._

---

## Síntesis formativa

En este repaso 3 de la **Clase 7** dominamos las herramientas fundamentales del forecasting moderno:

1. **Estructura y datos:** entendimos el formato largo estándar (`unique_id`, `ds`, `y`) que permite
   procesar cientos de series en paralelo.
2. **Anatomía visual:** aprendimos a reconocer tendencia, estacionalidad y quiebres estructurales antes
   de escribir una sola línea de modelado.
3. **Líneas base:** incorporamos la regla de oro: ningún modelo complejo se aprueba si no supera a
   `Naive` o `SeasonalNaive`.
4. **Modelos automáticos:** aprovechamos `AutoETS` y `AutoARIMA` en StatsForecast para ajustar modelos
   robustos de forma ágil (*low-code*).
5. **Validación honesta:** reemplazamos el k-fold aleatorio por **ventanas móviles temporales**
   (`cross_validation`), evitando la fuga temporal y evaluando con métricas tangibles (**MAE en pasajeros**
   y **MAPE en %**).
6. **Incertidumbre y decisiones:** comunicamos el pronóstico con **intervalos de predicción (80% y 95%)**
   y dimensionamos la capacidad operativa considerando la **asimetría de costos** entre quedarse corto y pasarse.
7. **Límites éticos:** distinguimos la predictibilidad legítima de la demanda frente al comportamiento
   estocástico de los activos bursátiles.

## Atribución de datos

**Tráfico aéreo de pasajeros en Chile** (`trafico_aereo_cl.csv`)

**Creador:** Junta de Aeronáutica Civil (JAC), Ministerio de Transportes y Telecomunicaciones de Chile.
**Fuente:** [Portal de Datos Abiertos del Gobierno de Chile](https://datos.gob.cl/dataset/trafico-aereo) y
estadísticas mensuales oficiales de la JAC.
**Licencia:** Datos públicos bajo [Licencia CC0 1.0 Universal](https://creativecommons.org/publicdomain/zero/1.0/).
**Modificación:** agregación a nivel mensual para tres series (`internacional`, `nacional`, `total`),
estandarización de nombres de columna al esquema `unique_id`, `ds`, `y` y codificación UTF-8 para uso docente.